In [1]:
!pip install -q pillow numpy scikit-image torch torchvision matplotlib tqdm

import os, random, time, zipfile, io
import numpy as np
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Фиксация случайности для воспроизводимости (требование конкурса)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# ПУТЬ К ДАННЫМ (поменяй на свой или на путь в Colab)
DATA_DIR = r"D:\prog\python_p\ii\2p_2"
# DATA_DIR = "/content/data" # Раскомментируй для Google Colab

GRID, FS, IMG_SIZE = 24, 20, 480
TRAIN_INPUT_DIR = os.path.join(DATA_DIR, "train", "inputs")
TRAIN_TARGET_DIR = os.path.join(DATA_DIR, "train", "targets")
TEST_DIR = os.path.join(DATA_DIR, "test")
SUBMISSION_DIR = os.path.join(DATA_DIR, "submission_v2")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# НАСТРОЙКИ ДЛЯ МАКСИМАЛЬНОГО SSIM
CONFIG = {
    "val_size": 500,
    "puzzle_max_images": 1000,   # Увеличили данные для пазла
    "puzzle_epochs": 8,          # Больше эпох
    "puzzle_batch_size": 128,
    "puzzle_lr": 1e-3,
    "puzzle_num_starts": 20,
    
    "restore_max_images": 1000,  # Увеличили данные для реставрации
    "restore_epochs": 15,        # Больше эпох для U-Net
    "restore_batch_size": 4,     # 4 безопасно для 16GB VRAM
    "restore_lr": 1e-3,
    
    "eval_images": 5,
    "submission_num_starts": 30, # Больше попыток найти верный угол
}

# Поставь True только когда будешь готов делать финальный сабмит на 700 картинок!
# Пока оставь False, чтобы быстро проверить качество на 5 картинках.
RUN_FULL_SUBMISSION = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\qweqw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Device: cuda


C:\Users\qweqw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_image(path):
    return np.array(Image.open(path).convert("RGB"))

def get_png_names(directory):
    return sorted([n for n in os.listdir(directory) if n.lower().endswith(".png")])

def extract_fragments(img):
    frags = []
    for r in range(GRID):
        for c in range(GRID):
            frags.append(img[r*FS:(r+1)*FS, c*FS:(c+1)*FS])
    return np.array(frags, dtype=np.uint8)

def make_pair_canvas(frag_a, frag_b, orientation="right"):
    canvas = np.zeros((40, 40, 3), dtype=np.uint8)
    if orientation == "right":
        canvas[0:20, 0:20] = frag_a
        canvas[0:20, 20:40] = frag_b
    else:
        canvas[0:20, 0:20] = frag_a
        canvas[20:40, 0:20] = frag_b
    return canvas

def grid_to_canvas(fragments, grid):
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for r in range(GRID):
        for c in range(GRID):
            canvas[r*FS:(r+1)*FS, c*FS:(c+1)*FS] = fragments[grid[r, c]]
    return canvas

def calc_ssim(img1, img2):
    return ssim(img1, img2, channel_axis=2, data_range=255)

# === ТОЧНАЯ ИМИТАЦИЯ ИСКАЖЕНИЙ ПО УСЛОВИЮ ===
def corrupt_fragment(frag):
    x = frag.astype(np.float32)
    
    # 1. Контраст: 0.70 - 1.30
    contrast = np.random.uniform(0.70, 1.30)
    x = (x - 128.0) * contrast + 128.0
    
    # 2. Яркость: ±30
    x += np.random.uniform(-30.0, 30.0)
    x = np.clip(x, 0, 255).astype(np.uint8)
    
    # 3. Размытие: Gaussian 3x3 (radius=1.0 в PIL примерно дает 3x3 ядро)
    if np.random.random() < 0.95:
        x = np.array(Image.fromarray(x).filter(ImageFilter.GaussianBlur(radius=1.0)))
    
    # 4. JPEG: quality 35-50
    if np.random.random() < 0.95:
        quality = np.random.randint(35, 51)
        img = Image.fromarray(x)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        x = np.array(Image.open(buffer))
    
    # 5. Шум: σ = 40-55 (добавляем в конце)
    sigma = np.random.uniform(40.0, 55.0)
    noise = np.random.normal(0.0, sigma, x.shape).astype(np.float32)
    x = x.astype(np.float32) + noise
    x = np.clip(x, 0, 255).astype(np.uint8)
    
    return x

def make_dirty_assembled(clean_img):
    frags = extract_fragments(clean_img)
    dirty = np.zeros_like(clean_img)
    idx = 0
    for r in range(GRID):
        for c in range(GRID):
            dirty[r*FS:(r+1)*FS, c*FS:(c+1)*FS] = corrupt_fragment(frags[idx])
            idx += 1
    return dirty

In [3]:
input_names = get_png_names(TRAIN_INPUT_DIR)
target_names = get_png_names(TRAIN_TARGET_DIR)
test_names = get_png_names(TEST_DIR)

print(f"Train inputs: {len(input_names)}, targets: {len(target_names)}, test: {len(test_names)}")
assert len(input_names) == len(target_names) and set(input_names) == set(target_names)

all_names = input_names.copy()
random.shuffle(all_names)

val_names = all_names[:CONFIG["val_size"]]
train_names = all_names[CONFIG["val_size"]:]

print(f"Train split: {len(train_names)}, Val split: {len(val_names)}")

Train inputs: 7000, targets: 7000, test: 700
Train split: 6500, Val split: 500


In [4]:
class PairCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

class PuzzleDataset(Dataset):
    def __init__(self, names, target_dir, max_images=1000):
        self.frags_list = []
        self.items = []
        for name in names[:max_images]:
            img = load_image(os.path.join(target_dir, name))
            frags = extract_fragments(img)
            img_id = len(self.frags_list)
            self.frags_list.append(frags)
            
            pos_pairs, neg_pairs = [], []
            for r in range(GRID):
                for c in range(GRID):
                    idx = r * GRID + c
                    if c + 1 < GRID: pos_pairs.append((idx, idx + 1, 1, "right"))
                    if r + 1 < GRID: pos_pairs.append((idx, idx + GRID, 2, "below"))
            
            for _ in range(len(pos_pairs)):
                i1, i2 = random.sample(range(GRID * GRID), 2)
                neg_pairs.append((i1, i2, 0, random.choice(["right", "below"])))
                
            for i, j, label, orient in pos_pairs + neg_pairs:
                self.items.append((img_id, i, j, label, orient))
        random.shuffle(self.items)
        print(f"Пар для пазла: {len(self.items):,}")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        img_id, i, j, label, orient = self.items[idx]
        a, b = corrupt_fragment(self.frags_list[img_id][i]), corrupt_fragment(self.frags_list[img_id][j])
        x = torch.from_numpy(make_pair_canvas(a, b, orient).copy()).permute(2, 0, 1).float() / 255.0
        return x, int(label)

def train_puzzle(model, dataset, epochs, batch_size, lr):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(loader, desc=f"Puzzle ep {epoch+1}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
            pbar.set_postfix(loss=total_loss/total, acc=correct/total)
    return model

In [5]:
@torch.no_grad()
def compute_pair_scores(model, fragments, batch_size=2048):
    model.eval()
    n = len(fragments)
    right_scores = np.zeros((n, n), dtype=np.float32)
    below_scores = np.zeros((n, n), dtype=np.float32)
    
    for i in range(n):
        others = [j for j in range(n) if j != i]
        pairs_r = np.stack([make_pair_canvas(fragments[i], fragments[j], "right") for j in others])
        pairs_b = np.stack([make_pair_canvas(fragments[i], fragments[j], "below") for j in others])
        
        for pairs, scores_arr, label_idx in [(pairs_r, right_scores, 1), (pairs_b, below_scores, 2)]:
            x = torch.from_numpy(pairs).permute(0, 3, 1, 2).float() / 255.0
            probs = []
            for start in range(0, len(x), batch_size):
                batch = x[start:start + batch_size].to(device)
                probs.append(torch.softmax(model(batch), dim=1).cpu().numpy())
            probs = np.concatenate(probs, axis=0)
            scores_arr[i, others] = probs[:, label_idx]
    return right_scores, below_scores

def choose_start_candidates(right_scores, below_scores, num_starts=30):
    outgoing = right_scores.max(axis=1) + below_scores.max(axis=1)
    incoming = right_scores.max(axis=0) + below_scores.max(axis=0)
    return np.argsort(-(outgoing - incoming))[:num_starts]

def assemble_with_start_fast(right_scores, below_scores, start_fragment):
    n = right_scores.shape[0]
    grid = -np.ones((GRID, GRID), dtype=int)
    used = np.zeros(n, dtype=bool)
    grid[0, 0] = start_fragment
    used[start_fragment] = True
    
    for r in range(GRID):
        for c in range(GRID):
            if r == 0 and c == 0: continue
            scores = np.zeros(n, dtype=np.float32)
            count = 0
            if c > 0 and grid[r, c - 1] >= 0:
                scores += right_scores[grid[r, c - 1]]
                count += 1
            if r > 0 and grid[r - 1, c] >= 0:
                scores += below_scores[grid[r - 1, c]]
                count += 1
            if count > 0: scores /= count
            scores[used] = -1e18
            best_idx = int(np.argmax(scores))
            grid[r, c] = best_idx
            used[best_idx] = True
            
    total_score, total_count = 0.0, 0
    for r in range(GRID):
        for c in range(GRID):
            cur = grid[r, c]
            if c + 1 < GRID: total_score += right_scores[cur, grid[r, c+1]]; total_count += 1
            if r + 1 < GRID: total_score += below_scores[cur, grid[r+1, c]]; total_count += 1
    return grid, total_score / max(1, total_count)

def solve_puzzle(model, image, num_starts=30):
    fragments = extract_fragments(image)
    right_scores, below_scores = compute_pair_scores(model, fragments, batch_size=2048)
    candidates = choose_start_candidates(right_scores, below_scores, num_starts)
    
    best_grid, best_score = None, -1e18
    for start_id in candidates:
        grid, score = assemble_with_start_fast(right_scores, below_scores, int(start_id))
        if score > best_score:
            best_score, best_grid = score, grid
    return grid_to_canvas(fragments, best_grid)

In [6]:
class RestorationUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self._conv_block(3, 64)
        self.enc2 = self._conv_block(64, 128)
        self.enc3 = self._conv_block(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = self._conv_block(256, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(128, 64)
        self.out = nn.Conv2d(64, 3, kernel_size=1)
        
    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        e1, e2, e3 = self.enc1(x), self.enc2(self.pool(self.enc1(x))), self.enc3(self.pool(self.enc2(self.pool(self.enc1(x)))))
        # Исправленный forward для четкого соответствия размерностей
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.clamp(x + self.out(d1), 0.0, 1.0)

class RestorationDataset(Dataset):
    def __init__(self, names, target_dir, max_images=1000):
        self.names = names[:max_images]
        self.target_dir = target_dir
        print(f"Картинок для реставрации: {len(self.names)}")
    def __len__(self): return len(self.names)
    def __getitem__(self, idx):
        clean = load_image(os.path.join(self.target_dir, self.names[idx]))
        dirty = make_dirty_assembled(clean)
        return (torch.from_numpy(dirty.copy()).permute(2, 0, 1).float() / 255.0,
                torch.from_numpy(clean.copy()).permute(2, 0, 1).float() / 255.0)

def train_restorer(model, dataset, epochs, batch_size, lr):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss() # L1 лучше сохраняет детали и дает высокий SSIM
    for epoch in range(epochs):
        model.train()
        total_loss, total = 0.0, 0
        pbar = tqdm(loader, desc=f"Restorer ep {epoch+1}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            total += x.size(0)
            pbar.set_postfix(loss=total_loss/total)
    return model

def restore_image(model, img):
    model.eval()
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    x = x.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x)
    return np.clip(pred.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0, 0, 255).astype(np.uint8)

In [7]:
print("=== Обучение Пазла ===")
puzzle_dataset = PuzzleDataset(train_names, TRAIN_TARGET_DIR, CONFIG["puzzle_max_images"])
puzzle_model = PairCNN().to(device)
puzzle_model = train_puzzle(puzzle_model, puzzle_dataset, CONFIG["puzzle_epochs"], CONFIG["puzzle_batch_size"], CONFIG["puzzle_lr"])
torch.cuda.empty_cache()

print("\n=== Обучение Реставрации ===")
restore_dataset = RestorationDataset(train_names, TRAIN_TARGET_DIR, CONFIG["restore_max_images"])
restorer = RestorationUNet().to(device)
restorer = train_restorer(restorer, restore_dataset, CONFIG["restore_epochs"], CONFIG["restore_batch_size"], CONFIG["restore_lr"])
torch.cuda.empty_cache()

=== Обучение Пазла ===
Пар для пазла: 2,208,000


KeyboardInterrupt: 

In [ ]:
def evaluate_pipeline(puzzle_model, restorer, names, num_images=5, num_starts=20):
    scores = []
    for name in names[:num_images]:
        input_img = load_image(os.path.join(TRAIN_INPUT_DIR, name))
        target_img = load_image(os.path.join(TRAIN_TARGET_DIR, name))
        
        assembled = solve_puzzle(puzzle_model, input_img, num_starts)
        restored = restore_image(restorer, assembled)
        score = calc_ssim(target_img, restored)
        scores.append(score)
        print(f"{name}: SSIM = {score:.4f}")
        
    print(f"\n=== Mean Val SSIM: {np.mean(scores):.4f} ===")
    return np.mean(scores)

mean_ssim = evaluate_pipeline(puzzle_model, restorer, val_names, CONFIG["eval_images"], CONFIG["puzzle_num_starts"])

img_000434.png: SSIM = 0.2245
img_002772.png: SSIM = 0.2143
img_003401.png: SSIM = 0.3925
img_000173.png: SSIM = 0.0952
img_000354.png: SSIM = 0.2187

=== Mean Val SSIM: 0.2291 ===


In [ ]:
def make_submission(puzzle_model, restorer, test_names, out_dir, zip_path, full=False):
    os.makedirs(out_dir, exist_ok=True)
    
    # НЕ очищаем папку — сохраняем уже готовые файлы!
    already_done = set(os.listdir(out_dir))
    
    names = test_names if full else test_names[:5]
    names_to_process = [n for n in names if n not in already_done]
    
    print(f"Уже готово: {len(already_done)}")
    print(f"Осталось сделать: {len(names_to_process)}")
    
    for name in tqdm(names_to_process):
        img = load_image(os.path.join(TEST_DIR, name))
        assembled = solve_puzzle(puzzle_model, img, CONFIG["submission_num_starts"])
        restored = restore_image(restorer, assembled)
        Image.fromarray(restored).save(os.path.join(out_dir, name))
        
    # Пакуем ВСЁ в zip (и старое, и новое)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for filename in sorted(os.listdir(out_dir)):
            if filename.endswith(".png"):
                zf.write(os.path.join(out_dir, filename), arcname=filename)
    
    print(f"Готово: {zip_path}")
    print(f"Всего файлов в архиве: {len(os.listdir(out_dir))}")

Генерация сабмита для 5 изображений...


100%|██████████| 5/5 [01:00<00:00, 12.11s/it]

Готово: D:\prog\python_p\ii\2p_2\submission_v2.zip
Файлов в архиве: 5
